# SentinelGrafo — SOFTWARE DOCUMENTADO

**Clasificación de mensajes críticos en plataformas digitales mediante Redes Neuronales Gráficas**

---

## Índice del Software Documentado

1. Instalación de dependencias y montaje de Google Drive
2. Pipeline común: limpieza de texto y construcción de grafos
3. Carga y preprocesamiento: Caso 1 — Disaster Tweets
4. Carga y preprocesamiento: Caso 2 — Fake / Real News
5. Visualización de la topología de los grafos
6. Baseline: Logistic Regression sobre vectores TF-IDF
7. Definición de modelos GNN: GraphSAGE y GCN
8. Función unificada de entrenamiento y evaluación
9. Entrenamiento de modelos para ambos casos
10. Visualización 1: Curvas de entrenamiento (pérdida y accuracy)
11. Visualización 2: Matrices de confusión
12. Visualización 3: Proyección t-SNE de embeddings
13. Visualización 4: Diagrama esquemático de capas GNN (Message Passing)
14. Visualización 5: Gráfico de barras comparativo (Baseline vs GNN)
15. Dashboard interactivo — Demo de predicción en vivo
16. Tabla resumen final de resultados

---

## 1. Instalación de dependencias

Se instalan las bibliotecas necesarias:
- `torch_geometric` para GNN sobre grafos
- `ipywidgets` para el dashboard interactivo
- `seaborn` para visualizaciones estadísticas
- `scikit-learn` para TF-IDF, baseline y métricas
- `networkx` y `matplotlib` para visualización de grafos

In [ ]:
import sys
import subprocess

def install_package(package_name, alias=None):
    display_name = alias if alias else package_name
    print(f'Instalando {display_name}...')
    try:
        __import__(package_name.split('==')[0].split('>')[0].replace('-','_'))
        print(f'{display_name} ya esta instalado.')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name, '-q'])
        print(f'{display_name} instalado correctamente.')

install_package('torch_geometric', 'PyTorch Geometric')
install_package('ipywidgets', 'ipywidgets')
install_package('seaborn', 'Seaborn')
install_package('networkx', 'NetworkX')

print('\nTodas las dependencias listas.')

## 1b. Importación de todas las bibliotecas

Se centralizan todos los imports para mantener el notebook ordenado.

In [ ]:
import pandas as pd
import numpy as np
import re
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GCNConv
from torch_geometric.utils import to_networkx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
sns.set_style('whitegrid')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'PyTorch Geometric: {torch_geometric.__version__}')
print(f'Dispositivo: {DEVICE}')

## 2. Montaje de Google Drive y definición de rutas

Se monta Google Drive para acceder a los datasets almacenados.
Las rutas deben ajustarse según la ubicación de los archivos en tu Drive.

In [ ]:
import os

# Ruta base en local (ajusta segun tu maquina)
BASE_PATH = (
    r"C:/Users/JimXL/Desktop/proyecto de ia/"
    r"Redes Neuronales Graficas (GNN) - Inteligencia Artificial-20260629T230611Z-3-001/"
    r"Redes Neuronales Graficas (GNN) - Inteligencia Artificial"
)

RUTA_DISASTER_TRAIN = os.path.join(BASE_PATH, "03_Datasets", "disaster_tweets", "train.csv")
RUTA_TRUE_NEWS    = os.path.join(BASE_PATH, "03_Datasets", "fake_real_news", "DatosTrue.csv")
RUTA_FAKE_NEWS    = os.path.join(BASE_PATH, "03_Datasets", "fake_real_news", "DatosFake.csv")

for name, path in [("Disaster Train", RUTA_DISASTER_TRAIN),
                    ("True News", RUTA_TRUE_NEWS),
                    ("Fake News", RUTA_FAKE_NEWS)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"[{name}] Encontrado ({size_mb:.1f} MB): {path}")
    else:
        print(f"[{name}] NO ENCONTRADO: {path}")


## 3. Pipeline común — Limpieza de texto y construcción de grafos

Ambos casos de estudio comparten el mismo pipeline de preprocesamiento.
Se definen dos funciones reutilizables:

- **`clean_text(text)`** — Normaliza el texto: minúsculas, elimina URLs,
  HTML, puntuación y espacios sobrantes.
- **`build_graph(texts, labels, max_features, k)`** — Vectoriza con TF-IDF,
  construye el grafo con similitud coseno y k vecinos más cercanos,
  y empaqueta todo en un objeto `torch_geometric.data.Data` listo para la GNN.

In [ ]:
def clean_text(text):
    '''Limpia y normaliza un texto para el pipeline de NLP.'''
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zñáéíóúü\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_graph(texts, labels, max_features=3000, k=5):
    '''
    Construye un grafo textual a partir de una lista de textos.
    Retorna: (torch_geometric.data.Data, TfidfVectorizer, np.array TF-IDF)
    '''
    print(f'Vectorizando {len(texts)} textos con TF-IDF (max_features={max_features})...')
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    X_tfidf = vectorizer.fit_transform(texts).toarray()
    print(f'Matriz TF-IDF: {X_tfidf.shape}')

    print(f'Calculando similitud coseno y construyendo grafo k-NN (k={k})...')
    sim_matrix = cosine_similarity(X_tfidf)

    edge_src, edge_dst = [], []
    n = len(texts)
    for i in range(n):
        neighbors = np.argsort(sim_matrix[i])[-k-1:-1]
        for nb in neighbors:
            edge_src.append(i)
            edge_dst.append(nb)

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    x = torch.tensor(X_tfidf, dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y)
    print(f'Grafo construido: {data.num_nodes} nodos, {data.num_edges} aristas, '
          f'{data.num_features} features/nodo, {len(torch.unique(y))} clases')

    return data, vectorizer, X_tfidf

## 4. Caso 1 — Disaster Tweets: carga, balanceo y preprocesamiento

Se carga el dataset `train.csv` del desafío *Natural Language Processing with Disaster Tweets*.
Cada tweet tiene una etiqueta binaria:
- `target = 1` → el tweet describe un desastre real
- `target = 0` → el tweet NO describe un desastre real (uso figurado o irrelevante)

Se realiza balanceo de clases, limpieza de texto y split 80/20 para train/test.

In [ ]:
print('=' * 60)
print('CASO 1: DETECCION DE TWEETS SOBRE DESASTRES REALES')
print('=' * 60)

df_tweets = pd.read_csv(RUTA_DISASTER_TRAIN)
print(f'Tweets cargados: {len(df_tweets)}')
print(f'Columnas: {list(df_tweets.columns)}')
print(f'Distribucion de clases:\n{df_tweets["target"].value_counts()}')

df_tweets['clean_text'] = df_tweets['text'].apply(clean_text)
df_tweets = df_tweets[df_tweets['clean_text'].str.len() > 0].reset_index(drop=True)

min_class = df_tweets['target'].value_counts().min()
df_c1 = pd.concat([
    df_tweets[df_tweets['target'] == 0].sample(n=min_class, random_state=42),
    df_tweets[df_tweets['target'] == 1].sample(n=min_class, random_state=42)
], ignore_index=True)
df_c1 = df_c1.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nDataset balanceado Caso 1: {len(df_c1)} muestras')
print(f'Distribucion: {dict(df_c1["target"].value_counts())}')

texts_c1  = df_c1['clean_text'].tolist()
labels_c1 = df_c1['target'].tolist()

data_c1, vec_c1, tfidf_c1 = build_graph(texts_c1, labels_c1, max_features=3000, k=5)

idx_c1 = np.arange(len(df_c1))
train_idx_c1, test_idx_c1 = train_test_split(idx_c1, test_size=0.2,
                                            random_state=42, stratify=labels_c1)
train_mask_c1 = torch.zeros(data_c1.num_nodes, dtype=torch.bool)
test_mask_c1  = torch.zeros(data_c1.num_nodes, dtype=torch.bool)
train_mask_c1[train_idx_c1] = True
test_mask_c1[test_idx_c1]   = True
print(f'\nSplit Caso 1 — Train: {train_mask_c1.sum().item()}, Test: {test_mask_c1.sum().item()}')

## 5. Caso 2 — Fake / Real News: carga, balanceo y preprocesamiento

Se cargan los archivos `DatosTrue.csv` (noticias reales de Reuters) y `DatosFake.csv`
(noticias falsas). Se asigna `label=1` para noticias reales y `label=0` para noticias falsas.

Cada noticia se representa concatenando su `title` y `text` en un solo campo `content`.
Se muestrea 2000 noticias por clase (4000 totales) para mantener baja complejidad computacional.
Luego se limpia, vectoriza TF-IDF, construye el grafo k-NN y se hace split 80/20.

In [ ]:
print('=' * 60)
print('CASO 2: DETECCION DE NOTICIAS FALSAS VS REALES')
print('=' * 60)

MUESTRAS_POR_CLASE = 2000

print('Cargando archivos de noticias...')
df_true = pd.read_csv(RUTA_TRUE_NEWS, encoding='latin1')
df_fake = pd.read_csv(RUTA_FAKE_NEWS, encoding='latin1')
print(f'Noticias reales cargadas: {len(df_true)}')
print(f'Noticias falsas cargadas:  {len(df_fake)}')

df_true = df_true.sample(n=min(MUESTRAS_POR_CLASE, len(df_true)), random_state=42)
df_fake = df_fake.sample(n=min(MUESTRAS_POR_CLASE, len(df_fake)), random_state=42)

df_true['label'] = 1
df_fake['label'] = 0

df_c2 = pd.concat([df_true, df_fake], ignore_index=True)
df_c2 = df_c2.sample(frac=1, random_state=42).reset_index(drop=True)

df_c2['content'] = df_c2['title'].fillna('') + ' ' + df_c2['text'].fillna('')
df_c2['clean_content'] = df_c2['content'].apply(clean_text)
df_c2 = df_c2[df_c2['clean_content'].str.len() > 0].reset_index(drop=True)

print(f'\nDataset Caso 2: {len(df_c2)} noticias')
print(f'Distribucion de clases: {dict(df_c2["label"].value_counts())}')

texts_c2  = df_c2['clean_content'].tolist()
labels_c2 = df_c2['label'].tolist()

data_c2, vec_c2, tfidf_c2 = build_graph(texts_c2, labels_c2, max_features=3000, k=5)

idx_c2 = np.arange(len(df_c2))
train_idx_c2, test_idx_c2 = train_test_split(idx_c2, test_size=0.2,
                                            random_state=42, stratify=labels_c2)
train_mask_c2 = torch.zeros(data_c2.num_nodes, dtype=torch.bool)
test_mask_c2  = torch.zeros(data_c2.num_nodes, dtype=torch.bool)
train_mask_c2[train_idx_c2] = True
test_mask_c2[test_idx_c2]   = True
print(f'\nSplit Caso 2 — Train: {train_mask_c2.sum().item()}, Test: {test_mask_c2.sum().item()}')

## 6. Visualización de la topología de los grafos

Se visualiza un subconjunto de 800 nodos de cada grafo para apreciar la estructura
de conexiones generada por la similitud textual.

**Caso 1 (Disaster Tweets):** Naranja = Desastre real, Azul = No desastre
**Caso 2 (Fake/Real News):** Verde = Noticia real, Rojo = Noticia falsa

In [ ]:
def plot_graph_subset(data, labels_tensor, title, color_true, color_false,
                      label_true, label_false, max_nodes=800, seed=42):
    '''Visualiza un subgrafo de max_nodes nodos coloreados por clase.'''
    n = min(max_nodes, data.num_nodes)
    edge_index = data.edge_index
    mask = (edge_index[0] < n) & (edge_index[1] < n)
    sub_ei = edge_index[:, mask]
    sub_data = Data(x=data.x[:n], edge_index=sub_ei,
                    y=torch.tensor(labels_tensor[:n], dtype=torch.long))

    G = to_networkx(sub_data, to_undirected=True)
    y_np = labels_tensor[:n] if isinstance(labels_tensor, np.ndarray) else labels_tensor[:n].cpu().numpy()

    node_colors = [color_true if lbl == 1 else color_false for lbl in y_np]

    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=seed, k=0.15, iterations=30)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=50,
                           alpha=0.85, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.15, edge_color='gray', ax=ax)

    legend_elements = [
        mpatches.Patch(color=color_true, label=label_true),
        mpatches.Patch(color=color_false, label=label_false),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


print('\n>>> VISUALIZACION 1: TOPOLOGIA DE LOS GRAFOS <<<\n')

plot_graph_subset(data_c1, labels_c1,
                  'Caso 1: Grafo de Disaster Tweets (muestra de 800 nodos)',
                  color_true='#FF8C00', color_false='#1E90FF',
                  label_true='Desastre real', label_false='No desastre')

plot_graph_subset(data_c2, labels_c2,
                  'Caso 2: Grafo de Fake vs Real News (muestra de 800 nodos)',
                  color_true='#2E8B57', color_false='#DC143C',
                  label_true='Noticia real', label_false='Noticia falsa')

## 7. Baseline: Logistic Regression sobre vectores TF-IDF

Antes de entrenar las GNN, se establece un baseline con Regresión Logística
entrenada directamente sobre los vectores TF-IDF (sin información del grafo).

Esto permite medir cuánto aporta la estructura de grafo y el message passing de la GNN.

In [ ]:
def run_baseline(tfidf_matrix, labels, train_idx, test_idx, case_name):
    '''Entrena Logistic Regression sobre TF-IDF y retorna metricas.'''
    X_train = tfidf_matrix[train_idx]
    X_test  = tfidf_matrix[test_idx]
    y_train = np.array(labels)[train_idx]
    y_test  = np.array(labels)[test_idx]

    model = LogisticRegression(max_iter=2000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

    print(f'--- Baseline Logistic Regression: {case_name} ---')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print(classification_report(y_test, y_pred,
          target_names=['Clase 0', 'Clase 1']))

    return {'model': model, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1,
            'y_pred': y_pred, 'y_true': y_test}


print('\n>>> EVALUACION BASELINE <<<\n')

baseline_c1 = run_baseline(tfidf_c1, labels_c1, train_idx_c1, test_idx_c1,
                           'Caso 1: Disaster Tweets')
print()
baseline_c2 = run_baseline(tfidf_c2, labels_c2, train_idx_c2, test_idx_c2,
                           'Caso 2: Fake / Real News')

## 8. Definición de modelos GNN: GraphSAGE y GCN

Se implementa una clase `GNNClassifier` unificada que acepta un parámetro `model_type`
para seleccionar la arquitectura:

- **GraphSAGE (`sage`)**: Utiliza `SAGEConv` de PyG. Aprende una función de agregación
  sobre los vecinos muestreados. Es inductivo: puede generalizar a nodos no vistos.
- **GCN (`gcn`)**: Utiliza `GCNConv` de PyG. Aplica una convolución espectral
  de primer orden sobre todo el grafo.

Ambos modelos comparten la misma estructura: 2 capas de convolución de grafo con
activación ReLU, dropout para regularización, y una capa lineal final de clasificación.

El método `get_embeddings()` extrae las representaciones latentes de la penúltima capa,
útiles para la visualización t-SNE.

In [ ]:
class GNNClassifier(nn.Module):
    '''
    Clasificador basado en Graph Neural Networks.
    Soporta GraphSAGE y GCN como backbones.
    '''
    def __init__(self, in_channels, hidden_channels, out_channels,
                 model_type='sage', dropout=0.5):
        super().__init__()
        self.model_type = model_type
        self.dropout = dropout
        self.hidden_channels = hidden_channels

        if model_type == 'sage':
            self.conv1 = SAGEConv(in_channels, hidden_channels)
            self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        elif model_type == 'gcn':
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, hidden_channels)
        else:
            raise ValueError(f"Modelo '{model_type}' no soportado. Use 'sage' o 'gcn'.")

        self.classifier = nn.Linear(hidden_channels, out_channels)
        self._embeddings = None

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        self._embeddings = x
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.classifier(x)
        return x

    def get_embeddings(self):
        '''Retorna las embeddings de la penultima capa.'''
        return self._embeddings.detach().cpu().numpy()


print('Clase GNNClassifier definida correctamente.')
print('Arquitecturas soportadas: GraphSAGE (sage), GCN (gcn)')

## 9. Función unificada de entrenamiento y evaluación

`train_and_evaluate` entrena un modelo GNN con:
- Optimizador Adam con weight decay para regularización L2
- Cross-entropy loss
- Early stopping con paciencia configurable (restaura los mejores pesos)
- Registro de pérdida y accuracy por época para curvas de aprendizaje
- Extracción de embeddings finales para visualización t-SNE

In [ ]:
def train_and_evaluate(data, train_mask, test_mask, model_type='sage',
                       hidden_dim=64, epochs=200, lr=0.01, patience=30):
    '''
    Entrena un GNNClassifier y retorna modelo, historial, predicciones,
    embeddings y mejor accuracy en test.
    '''
    model = GNNClassifier(
        in_channels=data.num_features,
        hidden_channels=hidden_dim,
        out_channels=len(torch.unique(data.y)),
        model_type=model_type,
        dropout=0.5
    ).to(DEVICE)

    data = data.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'train_acc': [], 'test_acc': [], 'test_loss': []}
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    pbar_prefix = f'{model_type.upper()} '
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_mask], data.y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            train_pred = out_eval[train_mask].argmax(dim=1)
            train_acc = (train_pred == data.y[train_mask]).float().mean().item()
            test_pred = out_eval[test_mask].argmax(dim=1)
            test_acc = (test_pred == data.y[test_mask]).float().mean().item()
            test_loss_val = criterion(out_eval[test_mask], data.y[test_mask]).item()

        history['train_loss'].append(loss.item())
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['test_loss'].append(test_loss_val)

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f'{pbar_prefix}Early stopping en epoca {epoch+1}/{epochs}')
            break
    else:
        print(f'{pbar_prefix}Entrenamiento completo: {epochs} epocas')

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        test_pred = out[test_mask].argmax(dim=1).cpu().numpy()
        test_true = data.y[test_mask].cpu().numpy()
        embeddings = model.get_embeddings()

    results = {
        'model': model,
        'history': history,
        'y_pred': test_pred,
        'y_true': test_true,
        'embeddings': embeddings,
        'best_acc': best_test_acc,
    }
    return results


print('Funcion train_and_evaluate lista.')

## 10. Entrenamiento de modelos para ambos casos

Se entrena **GraphSAGE** y **GCN** para cada caso de estudio.
Todos los resultados se almacenan en diccionarios para su análisis posterior.

**Modelos a entrenar:**
- Caso 1 (Disaster Tweets): GraphSAGE + GCN
- Caso 2 (Fake/Real News): GraphSAGE + GCN

**Total: 4 modelos**

In [ ]:
print('=' * 60)
print('ENTRENAMIENTO DE MODELOS GNN')
print('=' * 60)

all_results = {}

# -----------------------------------------------------------
# Caso 1 — Disaster Tweets
# -----------------------------------------------------------
print('\n>>> CASO 1: Disaster Tweets — GraphSAGE <<<')
r_c1_sage = train_and_evaluate(data_c1, train_mask_c1, test_mask_c1,
                                model_type='sage', hidden_dim=64, epochs=200)
all_results['C1_GraphSAGE'] = r_c1_sage

print('\n>>> CASO 1: Disaster Tweets — GCN <<<')
r_c1_gcn = train_and_evaluate(data_c1, train_mask_c1, test_mask_c1,
                               model_type='gcn', hidden_dim=64, epochs=200)
all_results['C1_GCN'] = r_c1_gcn

# -----------------------------------------------------------
# Caso 2 — Fake / Real News
# -----------------------------------------------------------
print('\n>>> CASO 2: Fake / Real News — GraphSAGE <<<')
r_c2_sage = train_and_evaluate(data_c2, train_mask_c2, test_mask_c2,
                                model_type='sage', hidden_dim=64, epochs=200)
all_results['C2_GraphSAGE'] = r_c2_sage

print('\n>>> CASO 2: Fake / Real News — GCN <<<')
r_c2_gcn = train_and_evaluate(data_c2, train_mask_c2, test_mask_c2,
                               model_type='gcn', hidden_dim=64, epochs=200)
all_results['C2_GCN'] = r_c2_gcn

print('\n' + '=' * 60)
print('ENTRENAMIENTO COMPLETADO')
print('=' * 60)

for name, res in all_results.items():
    acc = res['best_acc']
    yp = res['y_pred']
    yt = res['y_true']
    _, _, f1, _ = precision_recall_fscore_support(yt, yp, average='macro')
    print(f'{name:20s} — Accuracy: {acc:.4f}  |  F1-macro: {f1:.4f}')

## 11. Visualización 1 — Curvas de entrenamiento

Se muestran las curvas de pérdida (loss) y accuracy durante el entrenamiento
para cada modelo. Esto permite diagnosticar overfitting, velocidad de convergencia
y comparar el comportamiento de GraphSAGE vs GCN.

In [ ]:
print('\n>>> VISUALIZACION 2: CURVAS DE ENTRENAMIENTO <<<\n')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

plot_configs = [
    (0, 0, 'C1_GraphSAGE', 'Caso 1: Disaster Tweets — GraphSAGE', '#FF8C00'),
    (0, 1, 'C1_GCN',       'Caso 1: Disaster Tweets — GCN',       '#1E90FF'),
    (1, 0, 'C2_GraphSAGE', 'Caso 2: Fake/Real News — GraphSAGE',  '#2E8B57'),
    (1, 1, 'C2_GCN',       'Caso 2: Fake/Real News — GCN',        '#DC143C'),
]

for row, col, key, title, color in plot_configs:
    h = all_results[key]['history']
    epochs = range(1, len(h['train_loss']) + 1)
    ax1 = axes[row, col]
    ax2 = ax1.twinx()

    ax1.plot(epochs, h['train_loss'], color=color, linestyle='-', linewidth=2,
             label='Train Loss')
    ax1.plot(epochs, h['test_loss'], color=color, linestyle='--', linewidth=2,
             label='Test Loss')
    ax2.plot(epochs, h['test_acc'], color='gray', linestyle='-.', linewidth=2,
             label='Test Accuracy')

    ax1.set_xlabel('Epoca', fontsize=10)
    ax1.set_ylabel('Loss', fontsize=10, color=color)
    ax1.tick_params(axis='y', labelcolor=color)
    ax2.set_ylabel('Accuracy', fontsize=10, color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    ax2.set_ylim(0, 1.05)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8)

    ax1.set_title(title, fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)

plt.suptitle('Curvas de Entrenamiento — Todos los Modelos',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 12. Visualización 2 — Matrices de confusión

Las matrices de confusión muestran cuántas predicciones acertó y falló cada modelo.
Se presentan en un grid 2x2 para comparación directa.

Para cada celda se incluye además el reporte de clasificación completo (precision, recall, F1).

In [ ]:
print('\n>>> VISUALIZACION 3: MATRICES DE CONFUSION <<<\n')

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

conf_labels_c1 = ['No desastre', 'Desastre real']
conf_labels_c2 = ['Noticia falsa', 'Noticia real']

conf_configs = [
    (0, 0, 'C1_GraphSAGE', 'Caso 1 — GraphSAGE', conf_labels_c1, 'Oranges'),
    (0, 1, 'C1_GCN',       'Caso 1 — GCN',       conf_labels_c1, 'Blues'),
    (1, 0, 'C2_GraphSAGE', 'Caso 2 — GraphSAGE', conf_labels_c2, 'Greens'),
    (1, 1, 'C2_GCN',       'Caso 2 — GCN',       conf_labels_c2, 'Reds'),
]

for row, col, key, title, labels, cmap in conf_configs:
    r = all_results[key]
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=axes[row, col],
                xticklabels=labels, yticklabels=labels, cbar=True,
                linewidths=0.5, linecolor='white')
    axes[row, col].set_title(title, fontsize=12, fontweight='bold')
    axes[row, col].set_xlabel('Prediccion', fontsize=10)
    axes[row, col].set_ylabel('Real', fontsize=10)

plt.suptitle('Matrices de Confusion — Todos los Modelos',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\nReportes de clasificacion detallados:\n')
for key, label_names in [('C1_GraphSAGE', conf_labels_c1),
                           ('C1_GCN', conf_labels_c1),
                           ('C2_GraphSAGE', conf_labels_c2),
                           ('C2_GCN', conf_labels_c2)]:
    r = all_results[key]
    print(f'--- {key} ---')
    print(classification_report(r['y_true'], r['y_pred'], target_names=label_names))
    print()

## 13. Visualización 3 — Proyección t-SNE de embeddings

Se extraen los embeddings de la penúltima capa de cada GNN (64 dimensiones)
y se proyectan a 2D usando t-SNE.

Esto permite observar si el modelo logra separar las clases en el espacio latente.
Los colores indican la clase real; los marcadores, la predicción del modelo.
Un círculo alrededor de un punto indica que el modelo acertó; una X indica error.

In [ ]:
print('\n>>> VISUALIZACION 4: PROYECCION t-SNE DE EMBEDDINGS <<<\n')

def plot_tsne(embeddings, y_true, y_pred, title, ax, colors):
    '''Proyecta embeddings con t-SNE y colorea por clase real + acierto/error.'''
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)//3))
    emb_2d = tsne.fit_transform(embeddings)

    correct = (y_true == y_pred)
    for cls_val, cls_name, col in zip([0, 1],
                                       ['Clase 0 (Correcto)', 'Clase 1 (Correcto)'],
                                       colors):
        mask_cls_correct = (y_true == cls_val) & correct
        if mask_cls_correct.any():
            ax.scatter(emb_2d[mask_cls_correct, 0], emb_2d[mask_cls_correct, 1],
                       c=col, marker='o', s=30, alpha=0.7, edgecolors='white',
                       linewidth=0.3, label=f'{cls_name}')

    for cls_val, col in zip([0, 1], colors):
        mask_error = (y_true == cls_val) & ~correct
        if mask_error.any():
            ax.scatter(emb_2d[mask_error, 0], emb_2d[mask_error, 1],
                       c=col, marker='X', s=60, alpha=0.9, edgecolors='black',
                       linewidth=0.5, label=f'Clase {cls_val} (Error)')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=7, loc='best')


fig, axes = plt.subplots(2, 2, figsize=(16, 14))

tsne_configs = [
    (0, 0, 'C1_GraphSAGE', 'Caso 1 — GraphSAGE Embeddings', ['#FF8C00', '#1E90FF']),
    (0, 1, 'C1_GCN',       'Caso 1 — GCN Embeddings',       ['#FF8C00', '#1E90FF']),
    (1, 0, 'C2_GraphSAGE', 'Caso 2 — GraphSAGE Embeddings', ['#DC143C', '#2E8B57']),
    (1, 1, 'C2_GCN',       'Caso 2 — GCN Embeddings',       ['#DC143C', '#2E8B57']),
]

for row, col, key, title, colors in tsne_configs:
    r = all_results[key]
    mask_test = test_mask_c1.numpy() if 'C1' in key else test_mask_c2.numpy()
    plot_tsne(r['embeddings'][mask_test], r['y_true'], r['y_pred'],
              title, axes[row, col], colors)

plt.suptitle('Visualizacion t-SNE de Embeddings de las GNN',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 14. Visualización 4 — Diagrama esquemático de capas GNN (Message Passing)

Se dibuja un diagrama conceptual del mecanismo de **Message Passing** que usan
GraphSAGE y GCN. El flujo es:

1. **Nodos de entrada** con features (TF-IDF, 3000 dims)
2. **AGGREGATE**: cada nodo recibe mensajes de sus vecinos
3. **UPDATE**: se combina la información propia con la agregada (MLP)
4. **Nueva representación** (embedding, 64 dims)
5. **Clasificador**: capa lineal que produce logits para cada clase

Esto se repite en 2 capas de convolución para propagar información a 2 saltos de distancia.

In [ ]:
print('\n>>> VISUALIZACION 5: DIAGRAMA DE CAPAS GNN (MESSAGE PASSING) <<<\n')

def draw_gnn_diagram():
    fig, ax = plt.subplots(1, 1, figsize=(16, 7))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 7)
    ax.axis('off')

    def draw_box(x, y, w, h, text, color, fontsize=10, text_color='white'):
        rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                                       facecolor=color, edgecolor='black',
                                       linewidth=1.5, alpha=0.9)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=fontsize, fontweight='bold', color=text_color)

    def draw_arrow(x1, y1, x2, y2, label='', color='gray'):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2,
                                    connectionstyle='arc3,rad=0'))
        if label:
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2 + 0.25
            ax.text(mx, my, label, ha='center', va='bottom',
                    fontsize=9, fontstyle='italic', color=color)

    y_center = 3.5

    draw_box(0.5, y_center-0.8, 2.8, 1.6,
             'NODOS DE\nENTRADA\n(features TF-IDF)', '#2C3E50', 10)

    draw_box(5.0, y_center-0.8, 2.8, 1.6,
             'AGGREGATE\n(Mensajes de\nvecinos k-NN)', '#2980B9', 10)

    draw_box(9.0, y_center-0.8, 2.8, 1.6,
             'UPDATE\n(MLP + ReLU\n+ Dropout)', '#27AE60', 10)

    draw_box(13.0, y_center-0.8, 2.8, 1.6,
             'EMBEDDINGS\n(64 dimensiones\nespacio latente)', '#8E44AD', 10)

    draw_box(6.5, 0.8, 4.5, 1.2,
             'CLASIFICADOR\nCapa Lineal -> Logits (2 clases)', '#E74C3C', 10)

    draw_arrow(3.3, y_center, 5.0, y_center, 'message', '#3498DB')
    draw_arrow(7.8, y_center, 9.0, y_center, 'passing', '#2ECC71')
    draw_arrow(11.8, y_center, 13.0, y_center, '', '#9B59B6')

    ax.annotate('', xy=(8.75, 1.8), xytext=(8.75, y_center-0.8),
                arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=2,
                                connectionstyle='arc3,rad=0'))

    ax.annotate('', xy=(8.75, 0.6), xytext=(8.75, 2.0),
                arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=2,
                                connectionstyle='arc3,rad=0.3'))
    ax.text(9.3, 1.3, 'x2 capas\n(2-hop\nmessage\npassing)',
            ha='left', va='center', fontsize=8, fontstyle='italic', color='#7F8C8D')

    ax.text(8, 6.3, 'DIAGRAMA DE MESSAGE PASSING EN GNN (GraphSAGE / GCN)',
            ha='center', va='center', fontsize=16, fontweight='bold', color='#2C3E50')
    ax.text(8, 5.6, 'Cada nodo agrega informacion de sus vecinos y actualiza su representacion',
            ha='center', va='center', fontsize=11, color='#7F8C8D')

    plt.tight_layout()
    plt.show()

draw_gnn_diagram()

## 15. Visualización 5 — Gráfico de barras comparativo

Comparación directa de Accuracy y F1-score entre:
- Baseline (Logistic Regression sobre TF-IDF)
- GraphSAGE (message passing sobre grafo k-NN)
- GCN (convolución espectral sobre grafo k-NN)

Para ambos casos de estudio.

In [ ]:
print('\n>>> VISUALIZACION 6: GRAFICO DE BARRAS COMPARATIVO <<<\n')

c1_acc_sage = all_results['C1_GraphSAGE']['best_acc']
c1_acc_gcn  = all_results['C1_GCN']['best_acc']
c2_acc_sage = all_results['C2_GraphSAGE']['best_acc']
c2_acc_gcn  = all_results['C2_GCN']['best_acc']

_, _, c1_f1_sage, _ = precision_recall_fscore_support(
    all_results['C1_GraphSAGE']['y_true'], all_results['C1_GraphSAGE']['y_pred'],
    average='macro')
_, _, c1_f1_gcn, _ = precision_recall_fscore_support(
    all_results['C1_GCN']['y_true'], all_results['C1_GCN']['y_pred'],
    average='macro')
_, _, c2_f1_sage, _ = precision_recall_fscore_support(
    all_results['C2_GraphSAGE']['y_true'], all_results['C2_GraphSAGE']['y_pred'],
    average='macro')
_, _, c2_f1_gcn, _ = precision_recall_fscore_support(
    all_results['C2_GCN']['y_true'], all_results['C2_GCN']['y_pred'],
    average='macro')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, (case_name, baseline, sage_acc, gcn_acc, sage_f1, gcn_f1) in enumerate([
    ('Caso 1: Disaster Tweets', baseline_c1['acc'], c1_acc_sage, c1_acc_gcn,
     c1_f1_sage, c1_f1_gcn),
    ('Caso 2: Fake / Real News', baseline_c2['acc'], c2_acc_sage, c2_acc_gcn,
     c2_f1_sage, c2_f1_gcn),
]):
    ax = axes[ax_idx]
    x = np.arange(3)
    width = 0.35

    acc_values = [baseline, sage_acc, gcn_acc]
    f1_values  = [

    bars_acc = ax.bar(x - width/2, acc_values, width, label='Accuracy',
                      color=['#95A5A6', '#2C3E50', '#3498DB'], edgecolor='black')

    # Need baseline F1 - compute it
    _, _, bl_f1, _ = precision_recall_fscore_support(
        baseline_c1['y_true'] if ax_idx == 0 else baseline_c2['y_true'],
        baseline_c1['y_pred'] if ax_idx == 0 else baseline_c2['y_pred'],
        average='macro')

    f1_values = [bl_f1, sage_f1, gcn_f1]
    bars_f1 = ax.bar(x + width/2, f1_values, width, label='F1-score (macro)',
                     color=['#BDC3C7', '#7F8C8D', '#85C1E9'], edgecolor='black')

    ax.set_xticks(x)
    ax.set_xticklabels(['Baseline\n(Log. Reg.)', 'GraphSAGE', 'GCN'], fontsize=10)
    ax.set_ylabel('Puntuacion', fontsize=11)
    ax.set_title(case_name, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars_acc, acc_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar, val in zip(bars_f1, f1_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Comparativa: Baseline vs GraphSAGE vs GCN',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 16. Dashboard interactivo — Demo de predicción en vivo

**Interfaz de usuario final del software SentinelGrafo.**

Permite:
1. Seleccionar el caso de estudio (Disaster Tweets o Fake News)
2. Ingresar un texto libre en inglés
3. Predecir la clase con el modelo GraphSAGE entrenado
4. Visualizar el nivel de confianza de la predicción
5. Ver los 3 vecinos más similares del dataset de entrenamiento

**Cómo funciona:** El nuevo texto se limpia, se vectoriza con TF-IDF,
se conecta al grafo existente mediante k-NN, y se ejecuta el forward pass
de la GNN para obtener la predicción del nuevo nodo.

In [ ]:
def predict_new_text(raw_text, model, data, vectorizer, tfidf_matrix, labels,
                     texts_original, case_name, k=5):
    '''
    Predice la clase de un nuevo texto usando la GNN entrenada.
    Retorna: (prediccion, confianza, vecinos_info)
    '''
    clean = clean_text(raw_text)
    if len(clean) < 3:
        return None, None, 'El texto es demasiado corto. Ingresa al menos 3 caracteres.'

    new_vec = vectorizer.transform([clean]).toarray()

    sims = cosine_similarity(new_vec, tfidf_matrix)[0]
    neighbor_idx = np.argsort(sims)[-k:][::-1]

    new_x = torch.tensor(new_vec, dtype=torch.float)
    all_x = torch.cat([data.x.cpu(), new_x], dim=0)

    new_src = [data.num_nodes] * k
    new_dst = neighbor_idx.tolist()
    all_edges = torch.cat([
        data.edge_index.cpu(),
        torch.tensor([new_src, new_dst], dtype=torch.long),
        torch.tensor([new_dst, new_src], dtype=torch.long)
    ], dim=1)

    model.eval()
    with torch.no_grad():
        out = model(all_x.to(DEVICE), all_edges.to(DEVICE))
        probs = F.softmax(out[data.num_nodes], dim=0).cpu().numpy()
        pred_class = int(probs.argmax())
        confidence = float(probs.max())

    neighbors_info = []
    for idx in neighbor_idx:
        similarity = float(sims[idx])
        label_str = 'REAL' if labels[idx] == 1 else 'FALSO'
        snippet = texts_original[idx][:100] + '...' if len(texts_original[idx]) > 100 else texts_original[idx]
        neighbors_info.append({
            'similarity': similarity,
            'label': label_str,
            'snippet': snippet
        })

    return pred_class, confidence, neighbors_info


# ---- Dashboard UI ----
print('\n>>> DASHBOARD INTERACTIVO <<<\n')

case_selector = widgets.Dropdown(
    options=[
        ('Caso 1: Disaster Tweets', 'c1'),
        ('Caso 2: Fake / Real News', 'c2'),
    ],
    value='c1',
    description='Caso:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

text_input = widgets.Textarea(
    placeholder='Escribe un tweet o noticia en ingles para clasificar...',
    description='Texto:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%', height='100px')
)

predict_btn = widgets.Button(
    description='PREDECIR',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='150px', height='40px')
)

output_area = widgets.Output()

def infer_case_data(case_key):
    if case_key == 'c1':
        return (all_results['C1_GraphSAGE']['model'], data_c1, vec_c1, tfidf_c1,
                np.array(labels_c1), df_c1['text'].tolist())
    else:
        return (all_results['C2_GraphSAGE']['model'], data_c2, vec_c2, tfidf_c2,
                np.array(labels_c2), df_c2['content'].tolist())

def on_predict_clicked(b):
    with output_area:
        clear_output(wait=True)
        raw_text = text_input.value.strip()
        if not raw_text:
            display(HTML('<p style="color:#E74C3C;">Por favor ingresa un texto.</p>'))
            return

        model, data, vec, tfidf, labels, texts_orig = infer_case_data(case_selector.value)

        pred_class, confidence, info = predict_new_text(
            raw_text, model, data, vec, tfidf, labels, texts_orig, case_selector.value)

        if pred_class is None:
            display(HTML(f'<p style="color:#E74C3C;">{info}</p>'))
            return

        if case_selector.value == 'c1':
            class_names = {0: 'NO DESASTRE', 1: 'DESASTRE REAL'}
            color = '#E74C3C' if pred_class == 1 else '#3498DB'
        else:
            class_names = {0: 'NOTICIA FALSA', 1: 'NOTICIA REAL'}
            color = '#2E8B57' if pred_class == 1 else '#DC143C'

        display(HTML(f'''
            <div style="border:2px solid {color}; border-radius:12px; padding:16px;
                        margin:8px 0; background:#F8F9FA;">
                <h3 style="color:{color}; margin:0 0 8px 0;">
                    Resultado: {class_names[pred_class]}</h3>
                <p style="font-size:16px; margin:4px 0;">
                    Confianza: <b>{confidence:.1%}</b></p>
            </div>
        '''))

        if isinstance(info, list):
            display(HTML('<h4 style="margin-top:16px;">Vecinos mas similares en el grafo:</h4>'))
            for i, nb in enumerate(info):
                border_color = '#27AE60' if nb['label'] == 'REAL' else '#E74C3C'
                display(HTML(f'''
                    <div style="border-left:4px solid {border_color}; padding:8px 12px;
                                margin:4px 0; background:white; border-radius:4px;">
                        <b>Vecino #{i+1}</b> (similitud: {nb['similarity']:.3f})
                        — Etiqueta: <span style="color:{border_color}; font-weight:bold;">
                        {nb['label']}</span><br>
                        <span style="color:#666;">"{nb['snippet']}"</span>
                    </div>
                '''))

predict_btn.on_click(on_predict_clicked)

ui = widgets.VBox([
    widgets.HBox([case_selector]),
    text_input,
    predict_btn,
    output_area
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', border_radius='8px'))

display(HTML('<h3 style="margin-bottom:4px;">SentinelGrafo — Clasificador Interactivo</h3>'))
display(ui)

## 17. Tabla resumen final de resultados

Se consolidan todos los resultados en un DataFrame para comparación directa.
Incluye Baseline, GraphSAGE y GCN para ambos casos de estudio.

In [ ]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    return acc, prec, rec, f1


rows = []

for case_name, baseline, sage_key, gcn_key in [
    ('Caso 1: Disaster Tweets', baseline_c1, 'C1_GraphSAGE', 'C1_GCN'),
    ('Caso 2: Fake / Real News', baseline_c2, 'C2_GraphSAGE', 'C2_GCN'),
]:
    rows.append({
        'Caso': case_name,
        'Modelo': 'Baseline (Log. Reg.)',
        'Accuracy': f"{baseline['acc']:.4f}",
        'Precision': f"{baseline['prec']:.4f}",
        'Recall': f"{baseline['rec']:.4f}",
        'F1-score': f"{baseline['f1']:.4f}",
    })

    for model_name, key in [('GraphSAGE', sage_key), ('GCN', gcn_key)]:
        r = all_results[key]
        acc, prec, rec, f1 = compute_metrics(r['y_true'], r['y_pred'])
        rows.append({
            'Caso': case_name,
            'Modelo': model_name,
            'Accuracy': f"{acc:.4f}",
            'Precision': f"{prec:.4f}",
            'Recall': f"{rec:.4f}",
            'F1-score': f"{f1:.4f}",
        })

df_results = pd.DataFrame(rows)

def highlight_best(val, row, col):
    # Simple styling
    return ''

styled = df_results.style.set_caption('RESULTADOS FINALES — SentinelGrafo').set_properties(**{
    'text-align': 'center'
}).set_table_styles([
    dict(selector='caption', props=[('font-size', '16px'), ('font-weight', 'bold'),
                                    ('text-align', 'center'), ('padding', '10px')]),
    dict(selector='th', props=[('background-color', '#2C3E50'), ('color', 'white'),
                                ('font-weight', 'bold'), ('text-align', 'center')]),
    dict(selector='td', props=[('text-align', 'center'), ('padding', '8px')]),
])

display(styled)

print('\n>>> NOTA: Todas las metricas estan calculadas sobre el conjunto de TEST. <<<')
print('El grafo se construyo con k=5 vecinos y TF-IDF de 3000 features.')
print('Modelos GNN entrenados con 2 capas, 64 dimensiones ocultas y early stopping.')

---

## Software Documentado — Fin del Anexo

Este documento constituye el software documentado completo de **SentinelGrafo**,
desarrollado como parte del Trabajo Computacional Nº 1 del curso
Inteligencia Artificial 2026-1, Sección 3.

**Grupo Segovia — Integrantes:**
- Segovia Valencia Jim Bryan
- Morales Usca Andres
- Chavez Cerna Joshua

**Tema:** Redes Neuronales Gráficas: fundamentos, aplicación y casos de estudio

---